In [ ]:

import numpy as np


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.5 MB/s eta 0:00:00


In [ ]:
import torch
torch.cuda.is_available()

True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import io
import os
import random
from pathlib import Path

import requests
from PIL import Image

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import EfficientNet_B0_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print(torch.__version__)


2.11.0+cu128


In [ ]:
train_dir = Path(r"/content/drive/MyDrive/Car_Demage_Severity/training")
test_dir = Path(r"/content/drive/MyDrive/Car_Demage_Severity/validation")
save_dir = Path("/content/drive/MyDrive/yolov8_train_car/pkl")
save_dir.mkdir(parents=True, exist_ok=True)

model_path = save_dir / 'cnn_car.pkl'
seed=42
random.seed(seed)
torch.manual_seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


device(type='cuda')

In [ ]:
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights

# 1. Lấy weights và cấu hình transform chuẩn của EfficientNet-B2
weights = EfficientNet_B2_Weights.DEFAULT
base_eval_transform = weights.transforms()

# Định nghĩa kích thước ảnh đồng bộ cho B2 (Sử dụng 288 thay vì 320)
image_size = 288
resize_size = 320

mean = base_eval_transform.mean
std = base_eval_transform.std

# 2. Cấu hình Eval Transform
eval_transform = transforms.Compose([
    transforms.Resize((resize_size, resize_size)),
    transforms.CenterCrop((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

# 3. Cấu hình Train Transform (Tích hợp lại toàn bộ Data Augmentation cũ của bạn)
train_transform = transforms.Compose([
    transforms.Resize((resize_size, resize_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.12,
            contrast=0.12,
            saturation=0.08,
            hue=0.03,
        )
    ], p=0.5),
    transforms.RandomAffine(
        degrees=5,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05),
    ),
    transforms.CenterCrop((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

In [ ]:
full_train_for_train = datasets.ImageFolder(root=str(train_dir), transform=train_transform)
full_train_for_eval = datasets.ImageFolder(root=str(train_dir), transform=eval_transform)
test_dataset = datasets.ImageFolder(root=str(test_dir), transform=eval_transform)

if full_train_for_train.class_to_idx != test_dataset.class_to_idx:
    print("C?nh b?o: th? t? class train v? test kh?ng kh?p.")
    print("Train:", full_train_for_train.class_to_idx)
    print("Test :", test_dataset.class_to_idx)

targets = np.array(full_train_for_train.targets)
indices = np.arange(len(targets))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.25,
    random_state=seed,
    stratify=targets,
)

train_dataset = Subset(full_train_for_train, train_idx)
val_dataset = Subset(full_train_for_eval, val_idx)

class_names = full_train_for_train.classes
train_counts = np.bincount(targets[train_idx], minlength=len(class_names))
val_counts = np.bincount(targets[val_idx], minlength=len(class_names))

print("Classes:", class_names)
print(f"Train/Val/Test: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}")
print("Train class counts:", dict(zip(class_names, train_counts)))
print("Val class counts:", dict(zip(class_names, val_counts)))


Classes: ['01-minor', '02-moderate', '03-severe']
Train/Val/Test: 2209/737/390
Train class counts: {'01-minor': np.int64(807), '02-moderate': np.int64(693), '03-severe': np.int64(709)}
Val class counts: {'01-minor': np.int64(269), '02-moderate': np.int64(231), '03-severe': np.int64(237)}


In [ ]:
# DataLoader
# image_size=320 t?n VRAM h?n 224, n?n batch 16 th??ng an to?n h?n tr?n Colab/T4.
batch_size = 16
num_workers = 2
pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

print("?? kh?i t?o c?c DataLoader th?nh c?ng!")


?? kh?i t?o c?c DataLoader th?nh c?ng!


In [ ]:
train_targets = targets[train_idx]
class_counts = np.bincount(train_targets, minlength=len(class_names))

# D?ng tr?ng s? m?m ?? tr?nh l?p ?t m?u b? ??y qu? m?nh.
raw_class_weights = class_counts.sum() / (len(class_counts) * np.maximum(class_counts, 1))
soft_class_weights = np.sqrt(raw_class_weights)
soft_class_weights = soft_class_weights / soft_class_weights.mean()
class_weights = torch.tensor(soft_class_weights, dtype=torch.float32).to(device)


class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=1.5, label_smoothing=0.03):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = nn.functional.cross_entropy(
            logits,
            targets,
            weight=self.weight,
            label_smoothing=self.label_smoothing,
            reduction="none",
        )
        pt = torch.exp(-ce)
        loss = ((1.0 - pt) ** self.gamma) * ce
        return loss.mean()


criterion = FocalLoss(
    weight=class_weights,
    gamma=1.5,
    label_smoothing=0.03,
)

print("Class counts:", dict(zip(class_names, class_counts)))
print("Raw class weights:", raw_class_weights)
print("Soft class weights:", class_weights)


Class counts: {'01-minor': np.int64(807), '02-moderate': np.int64(693), '03-severe': np.int64(709)}
Raw class weights: [0.91243288 1.06253006 1.03855195]
Soft class weights: tensor([0.9536, 1.0290, 1.0174], device='cuda:0')


In [ ]:
@torch.no_grad()
def evaluate(model, loader, device, criterion=None):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_true = []
    all_pred = []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)

        if criterion is not None:
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)

        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

        all_true.extend(labels.cpu().numpy())
        all_pred.extend(preds.cpu().numpy())

    return {
        "loss": total_loss / max(total_samples, 1),
        "accuracy": total_correct / max(total_samples, 1),
        "macro_f1": f1_score(all_true, all_pred, average="macro", zero_division=0),
        "y_true": all_true,
        "y_pred": all_pred,
    }


def train_one_phase(
    model,
    train_loader,
    val_loader,
    device,
    epochs,
    optimizer,
    save_path,
    criterion,
    scheduler=None,
    save_metric="macro_f1",
    start_best=-1.0,
    patience=8,
    min_delta=1e-4,
    phase_name="",
):
    history = []
    best_score = start_best
    wait = 0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        total_samples = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            running_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_samples += labels.size(0)

        if scheduler is not None:
            scheduler.step()

        train_metrics = {
            "loss": running_loss / max(total_samples, 1),
            "accuracy": running_correct / max(total_samples, 1),
        }
        val_metrics = evaluate(model, val_loader, device, criterion)
        current_score = val_metrics[save_metric]

        if current_score > best_score + min_delta:
            best_score = current_score
            wait = 0
            torch.save(model.state_dict(), save_path)
        else:
            wait += 1

        history.append({
            "epoch": epoch,
            "train": train_metrics,
            "val": {
                "loss": val_metrics["loss"],
                "accuracy": val_metrics["accuracy"],
                "macro_f1": val_metrics["macro_f1"],
            },
        })

        prefix = f"{phase_name} " if phase_name else ""
        print(
            f"{prefix}Epoch {epoch:02d} | "
            f"train_loss={train_metrics['loss']:.4f} train_acc={train_metrics['accuracy']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.4f} "
            f"val_macro_f1={val_metrics['macro_f1']:.4f}"
        )

        if wait >= patience:
            print(f"Early stopping t?i epoch {epoch} v? {save_metric} kh?ng c?i thi?n sau {patience} epoch.")
            break

    print(f"Best {save_metric}: {best_score:.4f}")
    return history, best_score


In [ ]:
# 1. Khởi tạo mô hình EfficientNet-B2 chính xác
model = efficientnet_b2(weights=weights).to(device)

# 2. Lấy số features đầu vào của classifier B2 (là 1408 chứ không phải 1280 của B0)
num_ftrs = model.classifier[1].in_features

# 3. Định nghĩa lại lớp Classifier Head phù hợp với bài toán của bạn
model.classifier = nn.Sequential(
    nn.Dropout(p=0.40),
    nn.Linear(num_ftrs, len(class_names)),
).to(device)

# Phase 1: Đóng băng backbone, chỉ train đầu classifier head
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

optimizer_phase1 = torch.optim.AdamW(
    model.classifier.parameters(),
    lr=3e-4,
    weight_decay=1e-4,
)
scheduler_phase1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_phase1,
    T_max=20,
)

print("Phase 1: train EfficientNet-B2 classifier head")
# Khuyến khích giảm số epoch Phase 1 xuống 20-25 để tiết kiệm thời gian
history_phase1, best_score = train_one_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=25,
    optimizer=optimizer_phase1,
    save_path=model_path,
    criterion=criterion,
    scheduler=scheduler_phase1,
    save_metric="macro_f1",
    start_best=-1.0,
    patience=7,
    phase_name="Phase 1",
)

Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth


100%|██████████| 35.2M/35.2M [00:00<00:00, 182MB/s]


Phase 1: train EfficientNet-B2 classifier head
Phase 1 Epoch 01 | train_loss=0.4905 train_acc=0.5672 | val_loss=0.4222 val_acc=0.6798 val_macro_f1=0.6707
Phase 1 Epoch 02 | train_loss=0.3776 train_acc=0.6953 | val_loss=0.3668 val_acc=0.7273 val_macro_f1=0.7248
Phase 1 Epoch 03 | train_loss=0.3365 train_acc=0.7288 | val_loss=0.3394 val_acc=0.7273 val_macro_f1=0.7256
Phase 1 Epoch 04 | train_loss=0.3222 train_acc=0.7302 | val_loss=0.3280 val_acc=0.7436 val_macro_f1=0.7409
Phase 1 Epoch 05 | train_loss=0.3147 train_acc=0.7370 | val_loss=0.3164 val_acc=0.7571 val_macro_f1=0.7543
Phase 1 Epoch 06 | train_loss=0.3154 train_acc=0.7429 | val_loss=0.3098 val_acc=0.7449 val_macro_f1=0.7456
Phase 1 Epoch 07 | train_loss=0.3128 train_acc=0.7293 | val_loss=0.3105 val_acc=0.7531 val_macro_f1=0.7523
Phase 1 Epoch 08 | train_loss=0.2959 train_acc=0.7601 | val_loss=0.2996 val_acc=0.7666 val_macro_f1=0.7642
Phase 1 Epoch 09 | train_loss=0.3002 train_acc=0.7546 | val_loss=0.3009 val_acc=0.7626 val_macro_

In [ ]:
# Phase 2: fine-tune last EfficientNet blocks + classifier
# Lu?n b?t ??u phase 2 t? checkpoint t?t nh?t c?a phase 1.
model.load_state_dict(torch.load(model_path, map_location=device))

for param in model.parameters():
    param.requires_grad = False
for param in model.features[-2:].parameters():
    param.requires_grad = True
for param in model.classifier.parameters():
    param.requires_grad = True

optimizer_phase2 = torch.optim.AdamW([
    {"params": model.features[-2:].parameters(), "lr": 1e-5},
    {"params": model.classifier.parameters(), "lr": 5e-5},
], weight_decay=1e-4)
scheduler_phase2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_phase2,
    T_max=35,
)

print("Phase 2: fine-tune EfficientNet-B0 last blocks + classifier")
history_phase2, best_score = train_one_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=35,
    optimizer=optimizer_phase2,
    save_path=model_path,
    criterion=criterion,
    scheduler=scheduler_phase2,
    save_metric="macro_f1",
    start_best=best_score,
    patience=10,
    phase_name="Phase 2",
)

# Load best checkpoint and evaluate test
model.load_state_dict(torch.load(model_path, map_location=device))
test_metrics = evaluate(model, test_loader, device, criterion)

print("Test accuracy:", test_metrics["accuracy"])
print("Test macro_f1:", test_metrics["macro_f1"])

print(classification_report(
    test_metrics["y_true"],
    test_metrics["y_pred"],
    target_names=class_names,
    zero_division=0,
))

print(confusion_matrix(
    test_metrics["y_true"],
    test_metrics["y_pred"],
))


Phase 2: fine-tune EfficientNet-B0 last blocks + classifier
Phase 2 Epoch 01 | train_loss=0.2764 train_acc=0.7732 | val_loss=0.2841 val_acc=0.7748 val_macro_f1=0.7729
Phase 2 Epoch 02 | train_loss=0.2732 train_acc=0.7705 | val_loss=0.2707 val_acc=0.7788 val_macro_f1=0.7776
Phase 2 Epoch 03 | train_loss=0.2559 train_acc=0.7936 | val_loss=0.2789 val_acc=0.7734 val_macro_f1=0.7715
Phase 2 Epoch 04 | train_loss=0.2540 train_acc=0.7804 | val_loss=0.2741 val_acc=0.7856 val_macro_f1=0.7824
Phase 2 Epoch 05 | train_loss=0.2460 train_acc=0.7972 | val_loss=0.2672 val_acc=0.7843 val_macro_f1=0.7808
Phase 2 Epoch 06 | train_loss=0.2343 train_acc=0.8049 | val_loss=0.2637 val_acc=0.7992 val_macro_f1=0.7953
Phase 2 Epoch 07 | train_loss=0.2282 train_acc=0.8076 | val_loss=0.2647 val_acc=0.7788 val_macro_f1=0.7750
Phase 2 Epoch 08 | train_loss=0.2296 train_acc=0.8126 | val_loss=0.2669 val_acc=0.7897 val_macro_f1=0.7846
Phase 2 Epoch 09 | train_loss=0.2233 train_acc=0.8266 | val_loss=0.2492 val_acc=0.78

In [ ]:
# Predict from image URL
# L?u ?: model n?y train theo class_names c?a ImageFolder, v? d? 01-minor/02-moderate/03-severe.
def predict_from_url(url: str):
    resp = requests.get(url, timeout=20)
    resp.raise_for_status()

    image = Image.open(io.BytesIO(resp.content)).convert("RGB")
    x = eval_transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0].cpu()

    idx = int(torch.argmax(probs).item())
    return {
        "label": class_names[idx],
        "confidence": float(probs[idx].item()),
        "probs": {class_names[i]: float(probs[i].item()) for i in range(len(class_names))},
    }


In [ ]:
# Example URL prediction
# D?n URL ?nh xe h? h?ng v?o sample_url n?u mu?n test nhanh sau khi train xong.
sample_url = ""

if sample_url:
    result = predict_from_url(sample_url)
    print(result)
else:
    print("H?y g?n sample_url b?ng URL ?nh xe h? h?ng ?? test predict_from_url().")


H?y g?n sample_url b?ng URL ?nh xe h? h?ng ?? test predict_from_url().
